In [3]:
"""
Build script for the LangGraph Agent Evaluation & Regression Framework.

Paste this entire file into ONE Google Colab cell and run it (or run it
locally with `python build_agent_evaluation_project.py`). It writes the
full project to ./langgraph-agent-evaluation and, if running in Colab,
also creates /content/langgraph-agent-evaluation.zip.
"""
import os
import zipfile
from pathlib import Path

BASE_DIR = Path("langgraph-agent-evaluation")

FILES = {}

FILES[r"requirements.txt"] = r'''langgraph
sentence-transformers
numpy
pandas
pytest
'''

FILES[r"README.md"] = r'''# LangGraph Agent Evaluation & Regression Framework

A small LangGraph agent evaluated like a software system: correctness, tool
selection, trajectory, reliability, latency, and regression between two
agent versions.

## Project Overview

Pipeline: `USER INPUT → ROUTER → TOOL → ANSWER → EVALUATOR`

The router sends each input to **knowledge search**, a **calculator**, or a
**direct** reply. The evaluator then scores the full run, not just the final
text.

## Architecture
run_evaluation.py # entry point
src/agent.py # LangGraph agent (router, tools, answer)
src/metrics.py # correctness, accuracy, reliability, latency
src/evaluator.py # runs the dataset through an agent version
src/regression.py # V1 vs V2 comparison + regression decision
data/dataset.json # 12 evaluation cases
results/ # CSV/JSON outputs (generated on run)
tests/test_project.py # pytest suite


## Why Agent Evaluation Matters

An agent can produce the correct final answer while using the wrong tool or
wrong reasoning path. Therefore, evaluating only the final answer is
insufficient — this framework also checks *which* tool was used and *what
path* the agent took to get there.

## Tools

- **Knowledge Search** — keyword matching over 7 small hard-coded documents
  about LLMs, RAG, embeddings, agents, hallucinations, and evaluation. No
  vector database.
- **Calculator** — supports `+ - * /` via a small regex-based parser. No
  `eval()`.

## Metrics

| Metric | Method |
|---|---|
| Answer correctness | Exact numeric match for calculator cases; `all-MiniLM-L6-v2` semantic similarity for text cases |
| Tool accuracy | `expected_tool == actual_tool` |
| Trajectory accuracy | `expected_path == actual_path` |
| Reliability | successful cases / total cases |
| Latency | mean, median, p95 via `time.perf_counter()` |

## Trajectory Evaluation

Each run's node path (`router → tool → answer`) is recorded, and `evaluate`
is appended by the harness. A case only counts as trajectory-correct if the
full path matches exactly — catching cases where the right answer was
reached by the wrong route.

## V1 vs V2

- **V1**: correct router (calculator pattern checked before knowledge keywords).
- **V2**: the check order is swapped, so inputs like `"Evaluate 12 * 8"` match
  the knowledge keyword `"evaluat"` first and get misrouted to knowledge
  instead of the calculator.

## Regression Detection

```python
MAX_DROP = 0.05
```

If V1 → V2 causes answer correctness, tool accuracy, trajectory accuracy, or
reliability to drop by more than `MAX_DROP`, the run prints
`REGRESSION DETECTED`; otherwise `NO REGRESSION`. Latency is reported but not
used to trigger regression.

## Error Analysis

`results/error_analysis.csv` lists every failed case with `case_id`, `input`,
`expected_tool`, `actual_tool`, `expected_path`, `actual_path`,
`answer_score`, `latency`, and `failure_reason`.

## Installation

```bash
pip install -r requirements.txt
python run_evaluation.py
```

## Google Colab Usage

Run `build_agent_evaluation_project.py` in one cell — it creates the full
project under `/content/langgraph-agent-evaluation` and a matching zip. Then:

```bash
%cd /content/langgraph-agent-evaluation
!pip install -q -r requirements.txt
!python run_evaluation.py
```

The first run downloads the `all-MiniLM-L6-v2` model (~80 MB) from
Hugging Face; after that it is cached and the framework runs fully offline.

## Testing

```bash
pytest tests/ -q
```

Covers the calculator, knowledge search, routing (V1 and the V2 bug),
answer correctness, trajectory evaluation, and regression detection.

## Limitations

- Keyword matching is not real NLU; wording outside the 7 documents' keywords
  won't be found.
- Only 12 dataset cases — enough to demonstrate the framework, not a full
  benchmark.
- The semantic similarity model needs one internet-connected run to download
  its weights.

## Future Improvements

- Larger, more diverse dataset.
- Pluggable tools and a real vector-based retriever.
- Track more agent versions over time instead of just V1 vs V2.
'''

FILES[r"run_evaluation.py"] = r'''"""Entry point: run Agent V1 and Agent V2 over the dataset, evaluate, and check for regressions."""
import logging
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))

from src.evaluator import evaluate_agent, load_dataset, save_results  # noqa: E402
from src.regression import compare, save_report  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

DATA_PATH = ROOT / "data" / "dataset.json"
RESULTS_DIR = ROOT / "results"


def fmt(value: float) -> str:
    return f"{value:.3f}"


def print_block(name: str, metrics: dict) -> None:
    print(f"\n{name}")
    print(f"Answer Correctness: {fmt(metrics['answer_correctness'])}")
    print(f"Tool Accuracy:       {fmt(metrics['tool_accuracy'])}")
    print(f"Trajectory Accuracy: {fmt(metrics['trajectory_accuracy'])}")
    print(f"Reliability:         {fmt(metrics['reliability'])}")
    print(f"Mean Latency:        {metrics['mean_latency'] * 1000:.2f} ms")


def main() -> None:
    dataset = load_dataset(str(DATA_PATH))
    logger.info("Loaded %d evaluation cases", len(dataset))

    logger.info("Evaluating Agent V1 (correct router)...")
    v1 = evaluate_agent("v1", buggy=False, dataset=dataset)

    logger.info("Evaluating Agent V2 (buggy router)...")
    v2 = evaluate_agent("v2", buggy=True, dataset=dataset)

    save_results(str(RESULTS_DIR), v1, v2)

    report = compare(v1["metrics"], v2["metrics"])
    save_report(str(RESULTS_DIR), report)

    print("=== AGENT EVALUATION ===")
    print_block("V1", v1["metrics"])
    print_block("V2", v2["metrics"])
    print("\n=== REGRESSION ===")
    print(f"Status: {report['status']}")
    print(f"Reason: {report['reason']}")
    print(f"\nResults written to: {RESULTS_DIR}")


if __name__ == "__main__":
    main()
'''

FILES[r"src/agent.py"] = r'''"""LangGraph agent: router -> tool (knowledge/calculator/direct) -> answer.

Agent V1 uses a correct router. Agent V2 (buggy=True) swaps the order of the
routing checks, which causes some calculation inputs that also contain a
knowledge-like word (e.g. "Evaluate 12 * 8") to be misrouted to knowledge.
"""
from __future__ import annotations

import logging
import re
from typing import List, TypedDict

from langgraph.graph import END, StateGraph

logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Knowledge base: 7 small hard-coded documents, no vector DB.
# ---------------------------------------------------------------------------
KNOWLEDGE_DOCS = [
    {
        "id": "doc_llm",
        "keywords": ["llm", "large language model", "gpt"],
        "text": "A large language model (LLM) is a neural network trained on large "
        "amounts of text to predict and generate natural language.",
    },
    {
        "id": "doc_rag",
        "keywords": ["rag", "retrieval-augmented", "retrieval"],
        "text": "Retrieval-Augmented Generation (RAG) combines a language model with "
        "an external document retriever to ground answers in real data.",
    },
    {
        "id": "doc_embedding",
        "keywords": ["embedding", "embeddings", "vector"],
        "text": "An embedding is a numeric vector representation of text that captures "
        "semantic meaning so similar texts have similar vectors.",
    },
    {
        "id": "doc_agent",
        "keywords": ["agent", "agents", "autonomous", "tool use"],
        "text": "An AI agent is a system that plans, chooses tools, and takes actions "
        "to achieve a goal, going beyond a single question-answer exchange.",
    },
    {
        "id": "doc_hallucination",
        "keywords": ["hallucinat"],
        "text": "Hallucination is when a language model generates confident but "
        "factually incorrect or unsupported information.",
    },
    {
        "id": "doc_evaluation",
        "keywords": ["evaluat", "benchmark", "metric"],
        "text": "Evaluation of AI systems measures correctness, reliability, and "
        "behavior using metrics and test cases rather than opinion alone.",
    },
    {
        "id": "doc_finetuning",
        "keywords": ["fine-tun", "finetun", "training"],
        "text": "Fine-tuning adapts a pretrained model to a specific task by further "
        "training it on a smaller, task-specific dataset.",
    },
]

KNOWLEDGE_KEYWORDS = sorted({kw for d in KNOWLEDGE_DOCS for kw in d["keywords"]}, key=len, reverse=True)

CALC_PATTERN = re.compile(r"[-+]?\d+(?:\.\d+)?\s*[-+*/]\s*[-+]?\d+(?:\.\d+)?")


def knowledge_search(query: str) -> str:
    """Simple keyword-matching search over the hard-coded docs (no vector DB).

    Score = total character length of matched keywords, so a specific match
    (e.g. "hallucinat") outweighs a short, more generic one (e.g. "llm").
    """
    q = query.lower()
    best, best_score = KNOWLEDGE_DOCS[0], 0
    for doc in KNOWLEDGE_DOCS:
        score = sum(len(kw) for kw in doc["keywords"] if kw in q)
        if score > best_score:
            best, best_score = doc, score
    return best["text"]


def safe_calculate(expression: str) -> float:
    """Safe arithmetic for + - * / only. No eval()."""
    match = CALC_PATTERN.search(expression)
    if not match:
        raise ValueError(f"no arithmetic expression found in: {expression!r}")
    op_match = re.search(r"([-+]?\d+(?:\.\d+)?)\s*([-+*/])\s*([-+]?\d+(?:\.\d+)?)", match.group(0))
    a, op, b = float(op_match.group(1)), op_match.group(2), float(op_match.group(3))
    if op == "+":
        return a + b
    if op == "-":
        return a - b
    if op == "*":
        return a * b
    if op == "/":
        if b == 0:
            raise ZeroDivisionError("division by zero")
        return a / b
    raise ValueError(f"unsupported operator: {op}")  # pragma: no cover


class AgentState(TypedDict):
    input: str
    route: str
    tool_output: str
    answer: str
    path: List[str]


def make_router(buggy: bool = False):
    """Build the router node. buggy=True reproduces the Agent V2 routing bug."""

    def router_node(state: AgentState) -> AgentState:
        text = state["input"].lower()
        has_calc = bool(CALC_PATTERN.search(text))
        has_kw = any(kw in text for kw in KNOWLEDGE_KEYWORDS)

        if buggy:
            # BUG (V2): knowledge keywords are checked before the calculator
            # pattern, so a calc question containing a word like "evaluate"
            # gets misrouted to the knowledge tool.
            route = "knowledge" if has_kw else ("calculator" if has_calc else "direct")
        else:
            route = "calculator" if has_calc else ("knowledge" if has_kw else "direct")

        logger.debug("routed %r -> %s", state["input"], route)
        return {**state, "route": route, "path": state["path"] + ["router"]}

    return router_node


def knowledge_node(state: AgentState) -> AgentState:
    result = knowledge_search(state["input"])
    return {**state, "tool_output": result, "path": state["path"] + ["knowledge"]}


def calculator_node(state: AgentState) -> AgentState:
    try:
        result = safe_calculate(state["input"])
        output = str(int(result)) if result == int(result) else str(result)
    except (ValueError, ZeroDivisionError) as exc:
        output = f"error: {exc}"
    return {**state, "tool_output": output, "path": state["path"] + ["calculator"]}


_DIRECT_REPLIES = [
    (["hello", "hi "], "Hello! I'm a simple assistant here to help."),
    (["name"], "I don't have a personal name, I'm just an AI agent."),
    (["joke"], "I don't tell jokes, but I can search knowledge or do math."),
    (["thank"], "You're welcome!"),
]


def direct_node(state: AgentState) -> AgentState:
    text = state["input"].lower()
    reply = "I can help with knowledge questions or calculations."
    for triggers, response in _DIRECT_REPLIES:
        if any(t in text for t in triggers):
            reply = response
            break
    return {**state, "tool_output": reply, "path": state["path"] + ["direct"]}


def answer_node(state: AgentState) -> AgentState:
    return {**state, "answer": state["tool_output"], "path": state["path"] + ["answer"]}


def build_agent(buggy: bool = False):
    """Compile the LangGraph agent. buggy=True yields Agent V2."""
    graph = StateGraph(AgentState)
    graph.add_node("router", make_router(buggy))
    graph.add_node("knowledge", knowledge_node)
    graph.add_node("calculator", calculator_node)
    graph.add_node("direct", direct_node)
    graph.add_node("answer", answer_node)

    graph.set_entry_point("router")
    graph.add_conditional_edges(
        "router",
        lambda state: state["route"],
        {"knowledge": "knowledge", "calculator": "calculator", "direct": "direct"},
    )
    graph.add_edge("knowledge", "answer")
    graph.add_edge("calculator", "answer")
    graph.add_edge("direct", "answer")
    graph.add_edge("answer", END)
    return graph.compile()


def run_agent(app, user_input: str) -> AgentState:
    """Invoke the compiled graph on a single input and return the final state."""
    initial: AgentState = {"input": user_input, "route": "", "tool_output": "", "answer": "", "path": []}
    return app.invoke(initial)
'''

FILES[r"src/metrics.py"] = r'''"""Evaluation metrics: answer correctness, tool/trajectory accuracy, reliability, latency."""
from __future__ import annotations

import logging
import re
from typing import List, Optional, Sequence

import numpy as np

logger = logging.getLogger(__name__)

_model = None  # lazily-loaded SentenceTransformer, shared across calls


def _get_model():
    """Load the SentenceTransformer model once and cache it (downloaded on first use)."""
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer

        logger.info("Loading sentence-transformers model 'all-MiniLM-L6-v2'...")
        _model = SentenceTransformer("all-MiniLM-L6-v2")
    return _model


def semantic_similarity(actual: str, expected: str) -> float:
    """Cosine similarity between two texts using MiniLM embeddings, clamped to [0, 1]."""
    model = _get_model()
    emb = model.encode([actual, expected], normalize_embeddings=True)
    score = float(np.dot(emb[0], emb[1]))
    return max(0.0, min(1.0, score))


def _extract_number(text: str) -> Optional[float]:
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else None


def numeric_match(actual: str, expected: str, tol: float = 1e-6) -> float:
    """1.0 if the numeric value in `actual` matches `expected` within tolerance, else 0.0."""
    a, e = _extract_number(actual), _extract_number(expected)
    if a is None or e is None:
        return 0.0
    return 1.0 if abs(a - e) <= tol else 0.0


def answer_correctness(actual: str, expected: str, expected_tool: str) -> float:
    """Deterministic numeric comparison for calculator answers, semantic similarity otherwise."""
    if expected_tool == "calculator":
        return numeric_match(actual, expected)
    return semantic_similarity(actual, expected)


def tool_accuracy(expected_tool: str, actual_tool: str) -> bool:
    return expected_tool == actual_tool


def trajectory_accuracy(expected_path: Sequence[str], actual_path: Sequence[str]) -> bool:
    return list(expected_path) == list(actual_path)


def reliability(successes: int, total: int) -> float:
    return successes / total if total else 0.0


def latency_stats(latencies: List[float]) -> dict:
    """Mean, median and p95 latency (seconds) via numpy."""
    if not latencies:
        return {"mean": 0.0, "median": 0.0, "p95": 0.0}
    arr = np.array(latencies, dtype=float)
    return {
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "p95": float(np.percentile(arr, 95)),
    }
'''

FILES[r"src/evaluator.py"] = r'''"""Runs an agent version over the dataset and produces per-case + aggregate results."""
from __future__ import annotations

import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd

from src.agent import build_agent, run_agent
from src.metrics import answer_correctness, latency_stats, reliability, tool_accuracy, trajectory_accuracy

logger = logging.getLogger(__name__)

ANSWER_SCORE_THRESHOLD = 0.5  # minimum answer_score counted as a "successful" case


def load_dataset(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def evaluate_agent(version: str, buggy: bool, dataset: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Run every case through one agent version and collect per-case + aggregate metrics."""
    app = build_agent(buggy=buggy)
    rows = []
    successes = 0

    for case in dataset:
        start = time.perf_counter()
        failure_reason = ""
        try:
            result = run_agent(app, case["input"])
            actual_tool = result["route"]
            actual_answer = result["answer"]
            actual_path = result["path"] + ["evaluate"]  # evaluation step appended by the harness
            ran_ok = True
        except Exception as exc:  # noqa: BLE001 - agent failures are recorded, not raised
            logger.warning("Case %s raised an exception: %s", case["id"], exc)
            actual_tool, actual_answer, actual_path = "error", "", ["router", "error"]
            ran_ok = False
            failure_reason = f"exception: {exc}"

        latency = time.perf_counter() - start

        if ran_ok:
            score = answer_correctness(actual_answer, case["reference_answer"], case["expected_tool"])
            t_ok = tool_accuracy(case["expected_tool"], actual_tool)
            p_ok = trajectory_accuracy(case["expected_path"], actual_path)
            if not t_ok:
                failure_reason = "wrong tool"
            elif not p_ok:
                failure_reason = "wrong trajectory"
            elif score < ANSWER_SCORE_THRESHOLD:
                failure_reason = "low answer score"
            success = t_ok and p_ok and score >= ANSWER_SCORE_THRESHOLD
        else:
            score, t_ok, p_ok, success = 0.0, False, False, False

        successes += int(success)
        rows.append(
            {
                "agent_version": version,
                "case_id": case["id"],
                "input": case["input"],
                "expected_tool": case["expected_tool"],
                "actual_tool": actual_tool,
                "expected_path": " > ".join(case["expected_path"]),
                "actual_path": " > ".join(actual_path),
                "answer_score": round(score, 4),
                "tool_correct": t_ok,
                "trajectory_correct": p_ok,
                "latency": round(latency, 6),
                "success": success,
                "failure_reason": failure_reason,
            }
        )

    df = pd.DataFrame(rows)
    lat_stats = latency_stats(df["latency"].tolist())
    metrics = {
        "answer_correctness": float(df["answer_score"].mean()),
        "tool_accuracy": float(df["tool_correct"].mean()),
        "trajectory_accuracy": float(df["trajectory_correct"].mean()),
        "reliability": reliability(successes, len(dataset)),
        "mean_latency": lat_stats["mean"],
        "median_latency": lat_stats["median"],
        "p95_latency": lat_stats["p95"],
    }
    return {"rows": df, "metrics": metrics}


def save_results(results_dir: str, v1: Dict[str, Any], v2: Dict[str, Any]) -> None:
    """Write evaluation_results.csv (all cases, both versions) and error_analysis.csv."""
    out = Path(results_dir)
    out.mkdir(parents=True, exist_ok=True)

    all_rows = pd.concat([v1["rows"], v2["rows"]], ignore_index=True)
    all_rows.to_csv(out / "evaluation_results.csv", index=False)

    error_cols = [
        "agent_version",
        "case_id",
        "input",
        "expected_tool",
        "actual_tool",
        "expected_path",
        "actual_path",
        "answer_score",
        "latency",
        "failure_reason",
    ]
    errors = all_rows[~all_rows["success"]][error_cols]
    errors.to_csv(out / "error_analysis.csv", index=False)
'''

FILES[r"src/regression.py"] = r'''"""Compare Agent V1 vs Agent V2 aggregate metrics and detect regressions."""
from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any, Dict

import pandas as pd

logger = logging.getLogger(__name__)

MAX_DROP = 0.05

# 0-1 scaled metrics where (v1 - v2) > MAX_DROP counts as a regression.
DROP_METRICS = ["answer_correctness", "tool_accuracy", "trajectory_accuracy", "reliability"]
ALL_METRICS = DROP_METRICS + ["mean_latency"]


def compare(v1_metrics: Dict[str, float], v2_metrics: Dict[str, float]) -> Dict[str, Any]:
    """Compute per-metric deltas and decide REGRESSION DETECTED / NO REGRESSION."""
    rows = []
    regressed = []

    for key in ALL_METRICS:
        v1_val, v2_val = v1_metrics[key], v2_metrics[key]
        delta = v2_val - v1_val
        flagged = key in DROP_METRICS and (v1_val - v2_val) > MAX_DROP
        if flagged:
            regressed.append(key)
        rows.append(
            {
                "metric": key,
                "v1": round(v1_val, 4),
                "v2": round(v2_val, 4),
                "delta": round(delta, 4),
                "regressed": flagged,
            }
        )

    status = "REGRESSION DETECTED" if regressed else "NO REGRESSION"
    reason = (
        f"metric(s) dropped by more than {MAX_DROP}: {', '.join(regressed)}"
        if regressed
        else f"no metric dropped by more than {MAX_DROP}"
    )

    return {
        "status": status,
        "reason": reason,
        "max_drop_threshold": MAX_DROP,
        "v1_metrics": v1_metrics,
        "v2_metrics": v2_metrics,
        "comparison": rows,
    }


def save_report(results_dir: str, report: Dict[str, Any]) -> None:
    out = Path(results_dir)
    out.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(report["comparison"]).to_csv(out / "regression_comparison.csv", index=False)
    with open(out / "regression_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
'''

FILES[r"data/dataset.json"] = r'''[
  {
    "id": "case_001",
    "input": "What is Retrieval-Augmented Generation (RAG) used for?",
    "expected_tool": "knowledge",
    "expected_path": ["router", "knowledge", "answer", "evaluate"],
    "reference_answer": "Retrieval-Augmented Generation (RAG) combines a language model with an external document retriever to ground answers in real data."
  },
  {
    "id": "case_002",
    "input": "Can you explain what an embedding vector is used for?",
    "expected_tool": "knowledge",
    "expected_path": ["router", "knowledge", "answer", "evaluate"],
    "reference_answer": "An embedding is a numeric vector representation of text that captures semantic meaning so similar texts have similar vectors."
  },
  {
    "id": "case_003",
    "input": "Why do AI systems sometimes hallucinate false information?",
    "expected_tool": "knowledge",
    "expected_path": ["router", "knowledge", "answer", "evaluate"],
    "reference_answer": "Hallucination is when a language model generates confident but factually incorrect or unsupported information."
  },
  {
    "id": "case_004",
    "input": "What makes an AI agent different from a simple chatbot?",
    "expected_tool": "knowledge",
    "expected_path": ["router", "knowledge", "answer", "evaluate"],
    "reference_answer": "An AI agent is a system that plans, chooses tools, and takes actions to achieve a goal, going beyond a single question-answer exchange."
  },
  {
    "id": "case_005",
    "input": "Calculate 15 + 27",
    "expected_tool": "calculator",
    "expected_path": ["router", "calculator", "answer", "evaluate"],
    "reference_answer": "42"
  },
  {
    "id": "case_006",
    "input": "What is 144 / 12?",
    "expected_tool": "calculator",
    "expected_path": ["router", "calculator", "answer", "evaluate"],
    "reference_answer": "12"
  },
  {
    "id": "case_007",
    "input": "Evaluate 12 * 8",
    "expected_tool": "calculator",
    "expected_path": ["router", "calculator", "answer", "evaluate"],
    "reference_answer": "96"
  },
  {
    "id": "case_008",
    "input": "Evaluate 100 - 45",
    "expected_tool": "calculator",
    "expected_path": ["router", "calculator", "answer", "evaluate"],
    "reference_answer": "55"
  },
  {
    "id": "case_009",
    "input": "Hello, how are you today?",
    "expected_tool": "direct",
    "expected_path": ["router", "direct", "answer", "evaluate"],
    "reference_answer": "Hello! I'm a simple assistant here to help."
  },
  {
    "id": "case_010",
    "input": "What is your name?",
    "expected_tool": "direct",
    "expected_path": ["router", "direct", "answer", "evaluate"],
    "reference_answer": "I don't have a personal name, I'm just an AI agent."
  },
  {
    "id": "case_011",
    "input": "Can you tell me a joke?",
    "expected_tool": "direct",
    "expected_path": ["router", "direct", "answer", "evaluate"],
    "reference_answer": "I don't tell jokes, but I can search knowledge or do math."
  },
  {
    "id": "case_012",
    "input": "Thank you for your help today.",
    "expected_tool": "direct",
    "expected_path": ["router", "direct", "answer", "evaluate"],
    "reference_answer": "You're welcome!"
  }
]
'''

FILES[r"tests/test_project.py"] = r'''"""Tests for calculator, knowledge search, routing, correctness, trajectory, regression."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from src.agent import build_agent, run_agent, safe_calculate, knowledge_search
from src.metrics import numeric_match, tool_accuracy, trajectory_accuracy
from src.regression import compare


def test_calculator_operations():
    assert safe_calculate("15 + 27") == 42
    assert safe_calculate("100 - 45") == 55
    assert safe_calculate("12 * 8") == 96
    assert safe_calculate("144 / 12") == 12


def test_calculator_rejects_division_by_zero():
    try:
        safe_calculate("5 / 0")
        assert False, "expected ZeroDivisionError"
    except ZeroDivisionError:
        pass


def test_knowledge_search_matches_topic():
    result = knowledge_search("What is RAG used for?")
    assert "retrieval" in result.lower()

    result = knowledge_search("Why do models hallucinate?")
    assert "hallucinat" in result.lower()


def test_router_v1_calculator_case():
    app = build_agent(buggy=False)
    result = run_agent(app, "Evaluate 12 * 8")
    assert result["route"] == "calculator"
    assert result["path"] == ["router", "calculator", "answer"]


def test_router_v2_bug_misroutes_to_knowledge():
    app = build_agent(buggy=True)
    result = run_agent(app, "Evaluate 12 * 8")
    assert result["route"] == "knowledge"  # the intentional V2 bug


def test_router_direct_case():
    app = build_agent(buggy=False)
    result = run_agent(app, "Hello, how are you today?")
    assert result["route"] == "direct"


def test_numeric_answer_correctness():
    assert numeric_match("42", "42") == 1.0
    assert numeric_match("41", "42") == 0.0


def test_tool_accuracy():
    assert tool_accuracy("calculator", "calculator") is True
    assert tool_accuracy("calculator", "knowledge") is False


def test_trajectory_accuracy():
    path = ["router", "calculator", "answer", "evaluate"]
    assert trajectory_accuracy(path, path) is True
    assert trajectory_accuracy(path, ["router", "knowledge", "answer", "evaluate"]) is False


def test_regression_detection_flags_a_real_drop():
    v1 = {"answer_correctness": 1.0, "tool_accuracy": 1.0, "trajectory_accuracy": 1.0,
          "reliability": 1.0, "mean_latency": 0.002}
    v2 = {"answer_correctness": 0.8, "tool_accuracy": 0.8, "trajectory_accuracy": 0.8,
          "reliability": 0.8, "mean_latency": 0.002}
    report = compare(v1, v2)
    assert report["status"] == "REGRESSION DETECTED"


def test_regression_detection_allows_small_drop():
    v1 = {"answer_correctness": 1.0, "tool_accuracy": 1.0, "trajectory_accuracy": 1.0,
          "reliability": 1.0, "mean_latency": 0.002}
    v2 = {"answer_correctness": 0.98, "tool_accuracy": 1.0, "trajectory_accuracy": 1.0,
          "reliability": 1.0, "mean_latency": 0.002}
    report = compare(v1, v2)
    assert report["status"] == "NO REGRESSION"
'''

def main() -> None:
    # Create directory structure
    for sub in ["src", "data", "results", "tests"]:
        (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

    # Write every source/data/doc file
    for rel_path, content in FILES.items():
        target = BASE_DIR / rel_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")

    # Keep the empty results/ folder present until run_evaluation.py populates it
    (BASE_DIR / "results" / ".gitkeep").write_text("", encoding="utf-8")

    print(f"Project created at: {BASE_DIR.resolve()}")
    for p in sorted(BASE_DIR.rglob("*")):
        if p.is_file():
            print(" -", p.relative_to(BASE_DIR))

    # Zip the project. In Colab, also drop a copy at /content for easy download.
    zip_targets = [Path(f"{BASE_DIR.name}.zip")]
    if os.path.isdir("/content"):
        zip_targets.append(Path("/content") / f"{BASE_DIR.name}.zip")

    for zip_path in zip_targets:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in BASE_DIR.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=p.relative_to(BASE_DIR.parent))
        print(f"Zip created at: {zip_path.resolve()}")

    print("\nNext steps:")
    print(f"  cd {BASE_DIR.name}")
    print("  pip install -r requirements.txt")
    print("  python run_evaluation.py")


if __name__ == "__main__":
    main()

Project created at: /content/langgraph-agent-evaluation
 - README.md
 - data/dataset.json
 - requirements.txt
 - results/.gitkeep
 - run_evaluation.py
 - src/agent.py
 - src/evaluator.py
 - src/metrics.py
 - src/regression.py
 - tests/test_project.py
Zip created at: /content/langgraph-agent-evaluation.zip
Zip created at: /content/langgraph-agent-evaluation.zip

Next steps:
  cd langgraph-agent-evaluation
  pip install -r requirements.txt
  python run_evaluation.py


In [4]:
cd langgraph-agent-evaluation

/content/langgraph-agent-evaluation


In [5]:
pip install -r requirements.txt

In [7]:
!python run_evaluation.py

INFO: Loaded 12 evaluation cases
INFO: Evaluating Agent V1 (correct router)...
INFO: TensorFlow version 2.20.0 available.
INFO: JAX version 0.11.1 available.
INFO: Loading sentence-transformers model 'all-MiniLM-L6-v2'...
INFO: No device provided, using cpu
INFO: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
modules.json: 100% 349/349 [00:00<00:00, 745kB/s]
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolv

In [8]:
import shutil
from google.colab import files

# Change this to your folder name
folder = "/content/langgraph-agent-evaluation"

# Create ZIP
zip_path = shutil.make_archive(
    "/content/langgraph-agent-evaluation",
    "zip",
    folder
)

print("ZIP created:", zip_path)

# Download ZIP
files.download(zip_path)

ZIP created: /content/langgraph-agent-evaluation.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>